In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math

In [ ]:
pip install datasets tiktoken

In [ ]:
!pip install -U datasets huggingface_hub

In [ ]:
import datasets
import huggingface_hub

print(datasets.__version__)
print(huggingface_hub.__version__)

5.0.0
1.23.0


In [ ]:
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")

In [ ]:
dataset[0]

{'id': '0', 'translation': {'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'}}

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")

In [ ]:

PAD = enc.n_vocab
BOS = enc.n_vocab + 1
EOS = enc.n_vocab + 2

VOCAB_SIZE = enc.n_vocab + 3

In [ ]:
eng_text = dataset[0]['translation']['en']
fr_text = dataset[0]['translation']['fr']

eng_tokens = enc.encode(eng_text)
fr_tokens = enc.encode(fr_text)

print(eng_tokens)
print(fr_tokens)

[464, 22420, 11882]
[3123, 4490, 2185, 2518, 2516]


In [ ]:
eng_tokens = [BOS] + eng_tokens + [EOS]
fr_tokens = [BOS] + fr_tokens + [EOS]

print(eng_tokens)
print(fr_tokens)

[50258, 464, 22420, 11882, 50259]
[50258, 3123, 4490, 2185, 2518, 2516, 50259]


In [ ]:
MAX_LEN = 64

In [ ]:
def pad(ids):
    ids = ids[:MAX_LEN]
    ids += [PAD] * (MAX_LEN - len(ids))
    return ids

In [ ]:
eng_tokens = pad(eng_tokens)
fr_tokens = pad(fr_tokens)

print(len(eng_tokens))
print(len(fr_tokens))

64
64


In [ ]:
import torch
from torch.utils.data import Dataset

class TranslationDataset(Dataset):

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        sample = self.dataset[idx]

        src_text = sample["translation"]["en"]
        tgt_text = sample["translation"]["fr"]

        src = enc.encode(src_text)
        tgt = enc.encode(tgt_text)

        src = src + [EOS]
        tgt = [BOS] + tgt + [EOS]

        src = pad(src)
        tgt = pad(tgt)

        return (
            torch.tensor(src, dtype=torch.long),
            torch.tensor(tgt, dtype=torch.long)
        )

In [ ]:
train_dataset = TranslationDataset(dataset)

src, tgt = train_dataset[0]

print(src.shape)
print(tgt.shape)

torch.Size([64])
torch.Size([64])


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [ ]:
eng, fr = next(iter(train_loader))
tokens = []

for t in fr[0].tolist():
    if t == EOS:
        break
    if t < enc.n_vocab:
        tokens.append(t)

print(enc.decode(tokens))

-- Mais, dit vivement Harbert, il n'est pas possible qu'ils n'aient ni amadou, ni allumettes!


In [ ]:
emb = nn.Embedding(VOCAB_SIZE, 512)

eng_emb = emb(eng)
fr_emb = emb(fr)

print(eng_emb.shape)
print(fr_emb.shape)

torch.Size([32, 64, 512])
torch.Size([32, 64, 512])


In [ ]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
pos_encoding = PositionalEncoding(d_model=512)

eng, fr = next(iter(train_loader))

x = emb(eng)

print(x.shape)

torch.Size([32, 64, 512])


In [ ]:
x = pos_encoding(x)

print(x.shape)

torch.Size([32, 64, 512])


In [ ]:
# B = Batch size
# T = sequence lenght (64 here)
# C = dimension of embedding 512 here

# Understanding Cross-Attention in the Transformer Architecture

This breakdown walks through how the original Encoder-Decoder Transformer handles translation using a concrete example.

## The Setup
Suppose we want to translate:
* **English (Source):** `I love apples`
* **French (Target):** `J'aime les pommes`

---

## Step 1: The Encoder
The encoder reads the entire English sentence at once.

After passing through several encoder blocks, we get the final **Encoder Outputs**: `E1`, `E2`, and `E3`.

> 💡 **Think of these as "smart embeddings."** Because of self-attention, each vector now knows about the entire sentence. For example, `E2` ("love") knows that its subject is "I" and its object is "apples".

---

## Step 2: The Decoder Starts
The decoder generates the translation token-by-token.
* **Initial state:** `<BOS>` (Beginning of Sentence)
* **Step 2:** `<BOS> J'`
* **Step 3:** `<BOS> J' aime`
* **Current Step:** `<BOS> J' aime les` (We are now trying to predict the next word: `pommes`).

To do this, the Decoder layer runs through three distinct steps:

### 1. Masked Self-Attention ("What have I written so far?")
Before looking at the English text, the decoder looks only at itself to understand the current French context.
* **Input:** `<BOS> J' aime les`
* **Mechanism:** $Q = \text{Decoder}$, $K = \text{Decoder}$, $V = \text{Decoder}$

### 2. Cross-Attention ("Which English words should I look at?")
Now, the decoder needs to bridge the gap and look at the original English sentence to figure out what comes next. Because it is processing `les`, it needs to focus on `apples` in the encoder.

* **Query ($Q$) = Decoder:** The decoder is the one asking the question (*"I just wrote a plural article 'les', what entity does this map to?"*).
* **Keys ($K$) = Encoder:** The encoder provides the index/labels (*"Here is the context for 'I', 'love', and 'apples'"*).
* **Values ($V$) = Encoder:** The encoder holds the actual semantic information that the decoder wants to extract.



#### The Mathematical Intuition
When computing attention for the current decoder state ($D_3$ = `les`), the model compares its Query against all Encoder Keys:

$$\text{Query} (D_3) \times \text{Keys} (E_1, E_2, E_3)$$

1. **Calculate Raw Attention Scores:**
   * `I` $\rightarrow$ 0.1
   * `love` $\rightarrow$ 0.2
   * `apples` $\rightarrow$ 0.9

2. **Apply Softmax (Normalized Weights):**
   * `I` $\rightarrow$ 0.06
   * `love` $\rightarrow$ 0.12
   * `apples` $\rightarrow$ 0.82

3. **Weighted Sum of Values:**
   $$0.06 \times V(\text{I}) + 0.12 \times V(\text{love}) + 0.82 \times V(\text{apples})$$

The resulting vector contains mostly information about **"apples"**, making it incredibly easy for the model to predict **"pommes"** next.

> ❓ **Why not use the decoder for Key and Value?** Because the decoder doesn't know English! It only sees `<BOS> J' aime les`. The encoder is the only student that read and understood the source English sentence.

### 3. Feed Forward ("Process everything I know")
Now that the layer has gathered both the French grammatical context (from Self-Attention) and the relevant English information (from Cross-Attention), the Feed Forward network processes and transforms this combined representation so it can be passed up to predict the final token.

---

## Summary: The Decoder as Three Questions

Every single decoder layer is structurally organized to ask these three questions in order:

1. **Masked Self-Attention ($Q, K, V = \text{Decoder}$):** *"What is the structure of the target sentence I've written so far?"*
2. **Cross-Attention ($Q = \text{Decoder}, K,V = \text{Encoder}$):** *"What information from the source sentence matches what I'm currently writing?"*
3. **Feed Forward Network:** *"How do I combine these two pieces of information to pass it along to the next layer?"*

In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math

# 1. Single Attention Head Block
class Head(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.key   = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)
        self.head_size = head_size
        self.dropout = nn.Dropout(0.1)

    def forward(self, q, k, v, mask=None):
        B, T, C = q.shape  # Unpacks 3D tensor cleanly

        Q = self.query(q)
        K = self.key(k)
        V = self.value(v)

        # Compute affinities
        wei = Q @ K.transpose(-2, -1) * (self.head_size ** -0.5)
        if mask is not None:
            wei = wei.masked_fill(mask == 0, float("-inf"))

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ V
        return out

# 2. Multi-Head Attention Block
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, d_model):
        super().__init__()
        head_size = d_model // num_heads
        self.heads = nn.ModuleList([Head(d_model, head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, q, k, v, mask=None):
        # Fix positional bugs by naming parameters passed to Head
        out = torch.cat([h(q=q, k=k, v=v, mask=mask) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# 3. Feed Forward Block
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(0.1)
        )
    def forward(self, x):
        return self.net(x)

# 4. Corrected Encoder Layer Block
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads, d_model)
        self.ff = FeedForward(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        print("Encoder input:", x.shape)
        norm_x = self.norm1(x)
        print("After LayerNorm:", norm_x.shape)
        norm_x = self.norm1(x)
        x = x + self.attn(q=norm_x, k=norm_x, v=norm_x, mask=mask)
        x = x + self.ff(self.norm2(x))
        return x

# 5. Corrected Decoder Layer Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads, d_model)
        self.cross_attn = MultiHeadAttention(num_heads, d_model)
        self.ff = FeedForward(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, encoder_out, self_mask=None, cross_mask=None):
        norm_x = self.norm1(x)
        x = x + self.attn(q=norm_x, k=norm_x, v=norm_x, mask=self_mask)

        norm_x = self.norm2(x)
        x = x + self.cross_attn(q=norm_x, k=encoder_out, v=encoder_out, mask=cross_mask)

        x = x + self.ff(self.norm3(x))
        return x

# 6. Corrected Transformer Module
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_len=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = PositionalEncoding(d_model, max_len)

        self.encoder_layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads) for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_causal_mask(self, size, device):
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1) == 0
        return mask.float()

    def forward(self, src, tgt):
        device = src.device

        _, src_seq_len = src.shape
        _, tgt_seq_len = tgt.shape

        # FIX: Keep masks 3D (Batch, 1, Seq_Len) so they broadcast perfectly
        # against the (Batch, Seq_Len, Seq_Len) weights inside your isolated Heads.
        src_mask = (src != PAD).unsqueeze(1)

        tgt_causal = self.generate_causal_mask(tgt_seq_len, device)
        tgt_pad = (tgt != PAD).unsqueeze(1)

        # This will now safely combine into a 3D tensor: (Batch, Seq_Len, Seq_Len)
        tgt_mask = tgt_pad * tgt_causal

        src_features = self.pos_emb(self.token_emb(src))
        tgt_features = self.pos_emb(self.token_emb(tgt))

        # Pass through Encoder
        enc_out = src_features
        for layer in self.encoder_layers:
            enc_out = layer(enc_out, mask=src_mask)

        # Pass through Decoder
        dec_out = tgt_features
        for layer in self.decoder_layers:
            dec_out = layer(dec_out, enc_out, self_mask=tgt_mask, cross_mask=src_mask)

        return self.fc_out(dec_out)


In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math

# ==========================================
# 1. MODEL DEFINITIONS
# ==========================================

class Head(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.key   = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)
        self.head_size = head_size
        self.dropout = nn.Dropout(0.1)

    def forward(self, q, k, v, mask=None):
        B, T, C = q.shape  # Safely unpacks 3D tensor cleanly

        Q = self.query(q)
        K = self.key(k)
        V = self.value(v)

        # Compute affinities
        wei = Q @ K.transpose(-2, -1) * (self.head_size ** -0.5)

        if mask is not None:
            wei = wei.masked_fill(mask == 0, float("-inf"))

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ V
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, d_model):
        super().__init__()
        head_size = d_model // num_heads
        self.heads = nn.ModuleList([Head(d_model, head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, q, k, v, mask=None):
        # Explicitly pass named arguments to avoid positional mismatches
        out = torch.cat([h(q=q, k=k, v=v, mask=mask) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(0.1)
        )
    def forward(self, x):
        return self.net(x)

class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads, d_model)
        self.ff = FeedForward(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        norm_x = self.norm1(x)
        x = x + self.attn(q=norm_x, k=norm_x, v=norm_x, mask=mask)
        x = x + self.ff(self.norm2(x))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads, d_model)
        self.cross_attn = MultiHeadAttention(num_heads, d_model)
        self.ff = FeedForward(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, encoder_out, self_mask=None, cross_mask=None):
        norm_x = self.norm1(x)
        x = x + self.attn(q=norm_x, k=norm_x, v=norm_x, mask=self_mask)

        norm_x = self.norm2(x)
        x = x + self.cross_attn(q=norm_x, k=encoder_out, v=encoder_out, mask=cross_mask)

        x = x + self.ff(self.norm3(x))
        return x

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_len=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = PositionalEncoding(d_model, max_len)

        self.encoder_layers = nn.ModuleList([EncoderBlock(d_model, num_heads) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderBlock(d_model, num_heads) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_causal_mask(self, size, device):
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1) == 0
        return mask.float()

    def forward(self, src, tgt):
        device = src.device
        _, tgt_seq_len = tgt.shape

        # MASK GENERATION: Strictly 3D formatting to align with the list of isolated 3D Attention Heads
        src_mask = (src != PAD).unsqueeze(1)  # (Batch, 1, T_src)

        tgt_causal = self.generate_causal_mask(tgt_seq_len, device) # (T_tgt, T_tgt)
        tgt_pad = (tgt != PAD).unsqueeze(1) # (Batch, 1, T_tgt)
        tgt_mask = tgt_pad * tgt_causal # (Batch, T_tgt, T_tgt)

        src_features = self.pos_emb(self.token_emb(src))
        tgt_features = self.pos_emb(self.token_emb(tgt))

        enc_out = src_features
        for layer in self.encoder_layers:
            enc_out = layer(enc_out, mask=src_mask)

        dec_out = tgt_features
        for layer in self.decoder_layers:
            dec_out = layer(dec_out, enc_out, self_mask=tgt_mask, cross_mask=src_mask)

        return self.fc_out(dec_out)

# ==========================================
# 2. HYPERPARAMETERS & SETUP
# ==========================================

D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 3  # Lightweight starting configuration
EPOCHS = 10
LEARNING_RATE = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# ==========================================
# 3. INITIALIZE MODEL & OPTIMIZER
# ==========================================

model = Transformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    max_len=MAX_LEN
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# ==========================================
# 4. TRAINING LOOP
# ==========================================

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, (src, tgt) in enumerate(train_loader):
        src, tgt = src.to(device), tgt.to(device)

        # Safeguard: Squeeze accidental dimensions and enforce 2D formatting (B, T)
        if src.dim() > 2:
            src = src.view(src.size(0), -1)
        if tgt.dim() > 2:
            tgt = tgt.view(tgt.size(0), -1)

        # Teacher forcing split
        tgt_input = tgt[:, :-1]
        tgt_y = tgt[:, 1:]

        optimizer.zero_grad()

        # Forward pass
        output = model(src, tgt_input) # Output shape: (B, T-1, VOCAB_SIZE)

        # Reshape to calculate loss
        loss = criterion(
            output.reshape(-1, VOCAB_SIZE),
            tgt_y.reshape(-1)
        )

        # Backward pass
        loss.backward()

        # Gradient clipping to stabilize training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    print(f"===> Epoch {epoch+1} Completed! Average Loss: {total_loss / len(train_loader):.4f}")

Training on: cuda
Epoch [1/10] | Batch 0/3972 | Loss: 11.2348
Epoch [1/10] | Batch 100/3972 | Loss: 5.3692
Epoch [1/10] | Batch 200/3972 | Loss: 5.1988
Epoch [1/10] | Batch 300/3972 | Loss: 4.6304
Epoch [1/10] | Batch 400/3972 | Loss: 4.2360
Epoch [1/10] | Batch 500/3972 | Loss: 4.2587
Epoch [1/10] | Batch 600/3972 | Loss: 4.2109
Epoch [1/10] | Batch 700/3972 | Loss: 4.3408
Epoch [1/10] | Batch 800/3972 | Loss: 4.1403
Epoch [1/10] | Batch 900/3972 | Loss: 3.8493
Epoch [1/10] | Batch 1000/3972 | Loss: 3.6831
Epoch [1/10] | Batch 1100/3972 | Loss: 4.0013
Epoch [1/10] | Batch 1200/3972 | Loss: 3.7699
Epoch [1/10] | Batch 1300/3972 | Loss: 3.7181
Epoch [1/10] | Batch 1400/3972 | Loss: 3.4389
Epoch [1/10] | Batch 1500/3972 | Loss: 3.2292
Epoch [1/10] | Batch 1600/3972 | Loss: 3.6161
Epoch [1/10] | Batch 1700/3972 | Loss: 3.5243
Epoch [1/10] | Batch 1800/3972 | Loss: 3.4455
Epoch [1/10] | Batch 1900/3972 | Loss: 3.1226
Epoch [1/10] | Batch 2000/3972 | Loss: 3.4344
Epoch [1/10] | Batch 2100/3

In [40]:
import torch
import torch.nn.functional as F

def translate(model, sentence, enc, max_len=64, temperature=0.75, top_k=5):
    # Put model in evaluation mode (turns off dropout)
    model.eval()

    # 1. Tokenize and pad the input English sentence
    tokens = enc.encode(sentence) + [EOS]
    tokens = tokens[:max_len]
    tokens += [PAD] * (max_len - len(tokens))

    # Format as a 2D tensor: (Batch Size = 1, Sequence Length = 64)
    src = torch.tensor([tokens], dtype=torch.long, device=device)

    # Initialize the target sequence with the Beginning of Sentence token
    tgt_tokens = [BOS]

    # Generate tokens one by one
    with torch.no_grad():
        for _ in range(max_len):
            # Convert current target tokens to a 2D tensor: (1, current_len)
            tgt_tensor = torch.tensor([tgt_tokens], dtype=torch.long, device=device)

            # Forward pass through your trained Transformer
            outputs = model(src, tgt_tensor)

            # Pull the logits for the very last generated token -> shape: (VOCAB_SIZE)
            logits = outputs[0, -1, :] / temperature

            # Top-K filtering: keep only the top 5 most likely next words
            v, ix = torch.topk(logits, top_k)
            logits[logits < v[-1]] = -float('Inf')

            # Convert filtered logits into clean probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample a token from the probability distribution
            next_token = torch.multinomial(probs, num_samples=1).item()

            # Break the generation loop if the model predicts End of Sentence or Padding
            if next_token == EOS or next_token == PAD or len(tgt_tokens) >= max_len:
                break

            # Append the predicted token and repeat the loop
            tgt_tokens.append(next_token)

    # Post-processing: Remove structural control tokens (BOS, EOS, PAD) before decoding
    output_tokens = [t for t in tgt_tokens if t < enc.n_vocab]
    if not output_tokens:
        return "[No translation generated]"

    # Decode the token IDs back into a readable French string
    return enc.decode(output_tokens)

In [44]:
test_sentences = [
    "The Wanderer",
    "He looked at the map.",
    "Where is the boy?",
    "The large face of the sailor was happy.",
    "She walked along the road toward the house.",
    "Hello how are you doing",
    "Good day, how fares thy health"
]

print("=== FINAL TRANSLATION RESULTS ===\n")
for sentence in test_sentences:
    translation = translate(model, sentence, enc, max_len=64, temperature=0.7)
    print(f"English: {sentence}")
    print(f"French:  {translation}")
    print("-" * 40)

=== FINAL TRANSLATION RESULTS ===

English: The Wanderer
French:  Le grand Meaulnes
----------------------------------------
English: He looked at the map.
French:  Il regarda la carte.
----------------------------------------
English: Where is the boy?
French:  Où est le garçon ?
----------------------------------------
English: The large face of the sailor was happy.
French:  La grande figure de marin était heureuse.
----------------------------------------
English: She walked along the road toward the house.
French:  Elle marchait sur la route vers la maison.
----------------------------------------
English: Hello how are you doing
French:  Vous faites donc la faire?
----------------------------------------


In [45]:
import torch
from google.colab import files

# Save the weights
torch.save(model.state_dict(), 'transformer_translator.pt')

# Trigger the browser download
files.download('transformer_translator.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>